In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pickle

In [2]:
print(torch.cuda.is_available())

False


In [3]:
# Проверяем, доступны ли GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [9]:
# Загрузка DataFrame из файла
with open('dataset.pkl', 'rb') as f:
    df = pickle.load(f)

In [10]:
crystal = df['Crystal']
Crystal = []

for i in crystal:
    i = i[1:]
    foo, fooo = i.split(".")
    i = foo
    #print(i)
    Crystal.append(i)

In [6]:
Cr = df['Stats'].unique()
Cr

array([20000000.])

In [11]:
df['Crystal'] = Crystal
df = df.loc[((df['Crystal'] != 'Ag') & (df['Crystal'] != 'Au') & (df['Crystal'] != 'B4C') & (df['Crystal'] != 'H2O_ice_1h') & (df['Crystal'] != 'Hg') & (df['Crystal'] != 'Li')& (df['Crystal'] != 'LiF')& (df['Crystal'] != 'Gd'))]
#df = df.loc[((df['Stats'] != Cr[0]) & (df['Stats'] != Cr[1]))]
#df = df.reset_index(drop= True)
df

,Matrix,Crystal,Stats,Pulce duration
1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",Al,20000000.0,300
2,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",Al2O3_sapphire,20000000.0,300
5,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",Ba,20000000.0,300
6,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",Be,20000000.0,300
7,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",BeO,20000000.0,300
8,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",C_diamond,20000000.0,300
9,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",C_graphite,20000000.0,300
10,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",Cr,20000000.0,300
11,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",Cs,20000000.0,300
12,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",Cu,20000000.0,300


In [ ]:
maxStats = max(df['Stats'].unique())
minI = max(df['Pulce duration'].unique())

df_test = df.loc[((df['Stats'] == maxStats) & (df['Pulce duration'] == minI))]
df_test = df_test.reset_index(drop= True)

#segment_df_test = segment_df.loc[((segment_df['Stats'] == maxStats) & (segment_df['Pulce duration'] == minI))]
#segment_df_test = segment_df_test.reset_index(drop= True)

# Преобразование матриц в numpy массив перед преобразованием в тензоры
df_test_matrices = np.array(df_test['Matrix'].tolist())
#Segmenation_df_test_matrices = np.array(segment_df_test['Segmendation Diff'].tolist())

# Преобразование данных в тензоры PyTorch
df_test_tensor = torch.FloatTensor(df_test_matrices)
df_test_tensor = df_test_tensor.unsqueeze(1)

#Segmenation_df_test_tensor = torch.FloatTensor(Segmenation_df_test_matrices)
#Segmenation_df_test_tensor = Segmenation_df_test_tensor.unsqueeze(1)


#Segmenation_df_test_tensor.size()
df_test_tensor.size()

In [47]:
from skimage.transform import resize
# Функция для понижения разрешения с использованием skimage
def downsample_matrix(matrix, output_shape=(12, 24)):
    return resize(matrix, output_shape, mode='reflect', anti_aliasing=True)

def generate_mask(data, k):

    # Если данные не нормализованы, нормализуйте их
    data = data / np.max(data)

    # Устанавливаем порог на уровне 0.01 от максимального значения
    threshold = k * np.max(data)
    #threshold = 0.00000001 # Старое значение
    #threshold = 0.0001
    # Генерация маски: 1 для сигнала, 0 для фона
    mask = np.where(data > threshold, 1, 0)
    return mask

def split_matrix_into_16(matrix):
    """Функция для разделения матрицы на 16 равных частей."""
    h, w = matrix.shape
    patches = []
    
    # Шаги для разделения по высоте и ширине
    patch_height = h // 10  # 480 // 4 = 120
    patch_width = w // 10   # 250 // 4 = 62
    
    for i in range(0, h, patch_height):
        for j in range(0, w, patch_width):
            patch = matrix[i:i + patch_height, j:j + patch_width]
            patches.append(patch)
    
    return patches

def combine_patches_16(patches, original_shape=(250, 480)):
    """Сборка матрицы из 16 блоков."""
    h, w = original_shape
    patch_height = h // 10
    patch_width = w // 10
    new_matrix = np.zeros(original_shape)
    
    patch_idx = 0
    for i in range(0, h, patch_height):
        for j in range(0, w, patch_width):
            new_matrix[i:i + patch_height, j:j + patch_width] = patches[patch_idx]
            patch_idx += 1
            
    return new_matrix

def matrix_multiplication_with_threshold(A, B, threshold=0.1):
    # Применяем пороговое значение: если элемент меньше порога, он становится 0
    A = np.where(A < threshold, 0, A)
    
    # Умножаем матрицы
    result = A * B
    
    return result

In [48]:
# Пример понижения разрешения
matrices_high_res = df['Matrix'] # Исходные матрицы

mx_array = []

for matrix in matrices_high_res:
    paches = split_matrix_into_16(matrix)
    for mx in paches:
        #print(len(mx))
        mx_array.append(mx)

matrices_low_res = [downsample_matrix(matrix) for matrix in mx_array]

In [49]:
Downsample_train_tensor = torch.FloatTensor(np.array(matrices_low_res))
Downsample_train_tensor = Downsample_train_tensor.unsqueeze(1)
Diff_train_tensor = torch.FloatTensor(np.array(mx_array))
Diff_train_tensor = Diff_train_tensor.unsqueeze(1)
Downsample_train_tensor.size()

torch.Size([3000, 1, 12, 24])

In [53]:
Diff_train_tensor.size()

torch.Size([3000, 1, 25, 48])

In [70]:
# Пример понижения разрешения
matrices_high_res = df['Matrix'] # Исходные матрицы
matrices_low_res = [downsample_matrix(matrix) for matrix in matrices_high_res]
df['Compose Diffractions'] = matrices_low_res

In [71]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df.drop(['Matrix','Crystal', 'Stats', 'Pulce duration'], axis=1)

# Шаг 1: Преобразование строковых меток классов в числовые метки
label_encoder = LabelEncoder()

Diffractions = df.drop(['Crystal', 'Stats', 'Pulce duration', 'Compose Diffractions'], axis=1)

y = df['Crystal']

y_encoded = label_encoder.fit_transform(y)

Downsample_train, Downsample_test, Diff_train, Diff_test, y_train, y_test = train_test_split(X, Diffractions, y_encoded, test_size=0.1, random_state=42)

In [72]:
# Преобразование данных в тензоры PyTorch
Downsample_train_tensor = torch.FloatTensor(np.array(Downsample_train['Compose Diffractions'].tolist()))
Downsample_train_tensor = Downsample_train_tensor.unsqueeze(1)
Diff_train_tensor = torch.FloatTensor(np.array(Diff_train['Matrix'].tolist()))
Diff_train_tensor = Diff_train_tensor.unsqueeze(1)
y_train_tensor = torch.LongTensor(y_train)  # Используем LongTensor для целевых меток

Downsample_test_tensor = torch.FloatTensor(np.array(Downsample_test['Compose Diffractions'].tolist()))
Downsample_test_tensor = Downsample_test_tensor.unsqueeze(1)
Diff_test_tensor = torch.FloatTensor(np.array(Diff_test['Matrix'].tolist()))
Diff_test_tensor = Diff_test_tensor.unsqueeze(1)
y_test_tensor = torch.LongTensor(y_test)

In [73]:
Downsample_test_tensor.size()

torch.Size([3, 1, 125, 240])

In [74]:
Diff_test_tensor.size()

torch.Size([3, 1, 250, 480])

In [54]:
import torch.nn as nn
import torch.nn.functional as F

class SuperResolutionCNN(nn.Module):
    def __init__(self):
        super(SuperResolutionCNN, self).__init__()
        # Первый сверточный слой
        self.conv1 = nn.Conv2d(1, 64, kernel_size=9, padding=4)
        self.relu1 = nn.ReLU()
        
        # Второй сверточный слой
        self.conv2 = nn.Conv2d(64, 128, kernel_size=1)
        self.relu2 = nn.ReLU()

        # Второй сверточный слой
        self.conv3 = nn.Conv2d(128, 256, kernel_size=1)
        self.relu3 = nn.ReLU()

        # Второй сверточный слой
        self.conv4 = nn.Conv2d(256, 128, kernel_size=1)
        self.relu4 = nn.ReLU()

        # Второй сверточный слой
        self.conv5 = nn.Conv2d(128, 64, kernel_size=1)
        self.relu5 = nn.ReLU()

        #Второй сверточный слой
        self.conv6 = nn.Conv2d(64, 32, kernel_size=1)
        self.relu6 = nn.ReLU()
        
        # Третий сверточный слой
        self.conv7 = nn.Conv2d(32, 1, kernel_size=5, padding=2)
        
        # Слой увеличения разрешения
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.relu3(self.conv3(x))
        x = self.relu4(self.conv4(x))
        x = self.relu5(self.conv5(x))
        x = self.relu6(self.conv6(x))
        x = self.conv7(x)
        x = self.upsample(x)
        x = F.pad(x, (0, 0, 1, 0))  # (left, right, top, bottom)
        return x


class PH_UNet(nn.Module):
    def __init__(self):
        super(PH_UNet, self).__init__()
        self.encoder1 = self.conv_block(1, 64)
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)
        self.encoder4 = self.conv_block(256, 512)

        self.decoder1 = self.up_conv(512, 256)
        self.decoder2 = self.up_conv(256, 128)
        self.decoder3 = self.up_conv(128, 64)
        self.decoder4 = nn.Conv2d(64, 1, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def up_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            self.conv_block(out_channels, out_channels)
        )

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, 2))
        e3 = self.encoder3(F.max_pool2d(e2, 2))
        e4 = self.encoder4(F.max_pool2d(e3, 2))

        # Decoder
        d1 = self.decoder1(F.interpolate(e4, scale_factor=1, mode='bilinear', align_corners=True))
        d1 = F.pad(d1, (0, 0, 1, 0))  # (left, right, top, bottom)
        d2 = self.decoder2(F.interpolate(d1 + e3, scale_factor=1, mode='bilinear', align_corners=True))
        d3 = self.decoder3(F.interpolate(d2 + e2, scale_factor=1, mode='bilinear', align_corners=True))
        #d3 = F.pad(d3, (0, 0, 1, 0))  # (left, right, top, bottom)
        d4 = self.decoder4(F.interpolate(d3 + e1, scale_factor=1, mode='bilinear', align_corners=True))

        return torch.sigmoid(d4)

In [55]:
model = SuperResolutionCNN()

out1 = model(Downsample_train_tensor)

out1.size()

torch.Size([3000, 1, 25, 48])

In [80]:
patches = split_matrix_into_16(out[0][0].detach().numpy())

ph = torch.FloatTensor(patches)
ph = ph.unsqueeze(1)
ph.size()

torch.Size([16, 1, 125, 240])

In [104]:
model = PH_UNet()

out = model(out1)

out.size()

torch.Size([3, 1, 500, 960])

In [105]:
from torch.utils.data import TensorDataset, DataLoader, random_split

# Создание набора данных
train_dataset = TensorDataset(Downsample_train_tensor, Diff_train_tensor, y_train_tensor)

#test_dataset = TensorDataset(Downsample_test_tensor, Diff_test_tensor, y_test_tensor)

# Создание DataLoader для каждой выборки
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)

#test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)

In [107]:
# Пример настройки гиперпараметров и запуска обучения
model = SuperResolutionCNN().to(device)

criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.99))

#optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# Тренировка модели
num_epochs = 50
history = []

In [ ]:
for epoch in range(num_epochs):
    for inputs, targets, tags in train_loader:
        low_res = targets.to(device)
        high_res = model(low_res)

        for high_res_target in high_res:
            patches = split_matrix_into_16(high_res_target[0].cpu().detach().numpy())

            maps = []
            k_list = [k1, k2, k3, k4]

            for i in range(len(patches)):
                k = k_list[i // 4]
                map = generate_mask(patches[i], k)
                maps.append(map)

            maps = torch.FloatTensor(maps).to(device).unsqueeze(1)
            patches = torch.FloatTensor(patches).to(device).unsqueeze(1)

            optimizer.zero_grad()

            # Прямой проход
            outputs = model_UNET(patches)

            # Вычисление потерь
            loss = criterion(outputs, maps)

            # Обратное распространение
            loss.backward()
            optimizer.step()

        history.append(loss.item())

    print(f'Epoch {epoch+1}, Loss: {history[-1]:.2f}')


In [95]:
koef = [[0.02, 0.02, 0.07, 0.1], [0.03, 0.02, 0.01, 0.01], [0.05, 0.05, 0.05, 0.5], [0.03, 0.04, 0.2, 0.5], [0.1, 0.05, 0.2, 0.5], [0.04, 0.04, 0.02, 0.2], [0.04, 0.04, 0.3, 0.2], [0.01, 0.1, 0.3, 0.4], [0.04, 0.03, 0.01, 0.4], [0.04, 0.03, 0.1, 0.4], [0.05, 0.05, 0.03, 0.4], [0.04, 0.05, 0.25, 0.4], [0.06, 0.05, 0.1, 0.4], [0.04, 0.04, 0.1, 0.4], [0.05, 0.06, 0.03, 0.4], [0.08, 0.08, 0.03, 0.4], [0.1, 0.1, 0.05, 0.06], [0.04, 0.01, 0.1, 0.4], [0.05, 0.05, 0.07, 0.003], [0.04, 0.01, 0.01, 0.4], [0.05, 0.05, 0.2, 0.4], [0.05, 0.05, 0.1, 0.1], [0.05, 0.05, 0.1, 0.4], [0.1, 0.1, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1], [0.07, 0.05, 0.2, 0.4], [0.07, 0.05, 0.2, 0.4], [0.02, 0.02, 0.02, 0.4], [0.06, 0.02, 0.05, 0.4], [0.08, 0.05, 0.02, 0.04]]
Cris = df['Crystal'].unique()

koefs = dict(zip(label_encoder.transform(df['Crystal'].unique()), koef))

print(koefs)

{0: [0.02, 0.02, 0.07, 0.1], 1: [0.03, 0.02, 0.01, 0.01], 2: [0.05, 0.05, 0.05, 0.5], 3: [0.03, 0.04, 0.2, 0.5], 4: [0.1, 0.05, 0.2, 0.5], 5: [0.04, 0.04, 0.02, 0.2], 6: [0.04, 0.04, 0.3, 0.2], 7: [0.01, 0.1, 0.3, 0.4], 8: [0.04, 0.03, 0.01, 0.4], 9: [0.04, 0.03, 0.1, 0.4], 10: [0.05, 0.05, 0.03, 0.4], 11: [0.04, 0.05, 0.25, 0.4], 12: [0.06, 0.05, 0.1, 0.4], 13: [0.04, 0.04, 0.1, 0.4], 14: [0.05, 0.06, 0.03, 0.4], 15: [0.08, 0.08, 0.03, 0.4], 16: [0.1, 0.1, 0.05, 0.06], 17: [0.04, 0.01, 0.1, 0.4], 18: [0.05, 0.05, 0.07, 0.003], 19: [0.04, 0.01, 0.01, 0.4], 20: [0.05, 0.05, 0.2, 0.4], 21: [0.05, 0.05, 0.1, 0.1], 22: [0.05, 0.05, 0.1, 0.4], 23: [0.1, 0.1, 0.1, 0.1], 24: [0.1, 0.1, 0.1, 0.1], 25: [0.07, 0.05, 0.2, 0.4], 26: [0.07, 0.05, 0.2, 0.4], 27: [0.02, 0.02, 0.02, 0.4], 29: [0.06, 0.02, 0.05, 0.4], 28: [0.08, 0.05, 0.02, 0.04]}


In [ ]:
# Пример настройки гиперпараметров и запуска обучения
model_UNET = PH_UNet().to(device)

criterion = torch.nn.BCELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.99))

МЕГА ГИГАЧАД ТЕСТ КОЭФИЦИЕНТОВ ПАТЧЕЙ

In [ ]:
CR = label_encoder.transform(df_test['Crystal'])

CR = torch.LongTensor(CR)

df_test_dataset = TensorDataset(df_test_tensor, CR)

# Создание DataLoader для каждой выборки
df_test_loader = DataLoader(df_test_dataset, batch_size=5, shuffle=False)

Тест коэффициентов

In [ ]:
for targets, tags in df_test_loader:
    with torch.no_grad:
        target = targets.to(device)
        tags = tags.to(device)
        high_res = model(target)
    
    # Обработка каждого изображения в батче
    for idx, high_res_target in enumerate(high_res):
        # Получаем метку класса для текущего объекта
        tag = tags[idx].item()  # Извлекаем значение метки как int
        k_list = koefs[tag]  # Получаем соответствующий набор коэффициентов (k1, k2, k3, k4)

        # Разделение изображения на патчи
        patches = split_matrix_into_16(high_res_target[0].cpu().detach().numpy())

        maps = []
        
        # Генерация масок для каждого патча с использованием коэффициентов
        for i in range(len(patches)):
            k = k_list[i // 4]
            map = generate_mask(patches[i], k)
            maps.append(np.array(map))

        # Вывод всех патчей на график в сетке 4x4
        fig, axes = plt.subplots(4, 4, figsize=(12, 12))
        for i, ax in enumerate(axes.flat): 
            patch = maps[i]
            im = ax.imshow(patch, cmap='viridis', aspect='auto')
            ax.axis('off')

        # Показываем цветовую шкалу сбоку
        fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
        plt.show()


Тест юнета

In [ ]:
for targets, tags in df_test_loader:
    with torch.no_grad():
        target = targets.to(device)
        tags = tags.to(device)
        high_res = model(target)
    
    # Обработка каждого изображения в батче
    for idx, high_res_target in enumerate(high_res):
        # Получаем метку класса для текущего объекта
        tag = tags[idx].item()  # Извлекаем значение метки как int
        k_list = koefs[tag]  # Получаем соответствующий набор коэффициентов (k1, k2, k3, k4)

        # Разделение изображения на патчи
        patches = split_matrix_into_16(high_res_target[0].cpu().detach().numpy())

        patches = torch.FloatTensor(patches).to(device).unsqueeze(1)

        with torch.no_grad():
            out = model_UNET(patches)

        # Вывод всех патчей на график в сетке 4x4
        fig, axes = plt.subplots(4, 4, figsize=(12, 12))
        for i, ax in enumerate(axes.flat): 
            patch = out[i][0].cpu().detach().numpy()
            im = ax.imshow(patch, cmap='viridis', aspect='auto')
            ax.axis('off')

        # Показываем цветовую шкалу сбоку
        fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
        plt.show()

In [ ]:
import DiffQ as DQ

#Qplotter = DQ.Qwrapper(960, 500)

for targets, tags in df_test_loader:
    with torch.no_grad():
        target = targets.to(device)
        tags = tags.to(device)
        high_res = model(target)
    
    for i in range(len(high_res)):

        old = target[i][0].cpu().detach().numpy()

        # Разделение изображения на патчи
        patches = split_matrix_into_16(high_res[i][0].cpu().detach().numpy())

        patches = torch.FloatTensor(patches).to(device).unsqueeze(1)

        with torch.no_grad():
            out = model_UNET(patches)

        out = out.squeeze(1)

        # Создаем фигуру и оси
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(25, 8))

        Qplotter = DQ.Qwrapper(480, 250)

        q1, I1 = Qplotter.getQDIFADC(old, 1024)
        ax1.plot(q1, I1)
        # Первый график
        Diff = high_res[i][0].cpu().detach().numpy()
        #ax1.imshow(Diff, cmap='viridis', aspect='auto', norm='log')

        # Первый график
        Upscaled = combine_patches_16(out.cpu().detach().numpy())
        #ax2.imshow(Upscaled, cmap='viridis', aspect='auto',)

        Result = matrix_multiplication_with_threshold(Upscaled, Diff, threshold=0.3)
        #ax1.imshow(Result, cmap='viridis', aspect='auto', norm='log')

        Qplotter = DQ.Qwrapper(960, 500)

        q, I = Qplotter.getQDIFADC(Result, 1024)
        ax2.plot(q, I)

        #plt.figure(figsize=(10, 6))
        #plt.imshow(combine_patches_16(out.cpu().detach().numpy()), cmap='viridis', aspect='auto')
        plt.show()


Код обучения юнета на патчах

In [ ]:
for epoch in range(num_epochs):
    for inputs, targets, tags in train_loader:
        total_loss = 0  # Переменная для накопления потерь по всему батчу
        low_res = targets.to(device)
        tags = tags.to(device)  # Убедись, что метки передаются на правильное устройство
        high_res = model(low_res)

        # Обработка каждого изображения в батче
        for idx, high_res_target in enumerate(high_res):
            # Получаем метку класса для текущего объекта
            tag = tags[idx].item()  # Извлекаем значение метки как int
            k_list = koefs[tag]  # Получаем соответствующий набор коэффициентов (k1, k2, k3, k4)

            # Разделение изображения на патчи
            patches = split_matrix_into_16(high_res_target[0].cpu().detach().numpy())

            maps = []
            
            # Генерация масок для каждого патча с использованием коэффициентов
            for i in range(len(patches)):
                k = k_list[i // 4]
                map = generate_mask(patches[i], k)
                maps.append(np.array(map))

            # Преобразование активационных масок в тензоры
            masks = torch.FloatTensor(maps).to(device).unsqueeze(1)
            patches = torch.FloatTensor(patches).to(device).unsqueeze(1)

            #optimizer.zero_grad()

            # Прямой проход через UNET
            outputs = model_UNET(patches)

            # Вычисление потерь
            loss = criterion(outputs, masks)

            # Накопление потерь
            total_loss += loss

        # После всех изображений в батче выполняем обратное распространение
        optimizer.zero_grad()

        total_loss.backward()  # Обратное распространение накопленной ошибки
        optimizer.step()  # Обновление весов

        # Сохраняем среднюю потерю за батч
        history.append(total_loss.item() / len(train_loader))

    print(f'Epoch {epoch+1}, Loss: {history[-1]:.2f}')

Код обучения юнета для изображений размером 960 на 500 после склейки патчей

In [ ]:
for epoch in range(num_epochs):
    for inputs, targets, tags in train_loader:
        low_res = targets.to(device)
        tags = tags.to(device)  # Убедись, что метки передаются на правильное устройство
        high_res = model(low_res)

        activation_masks = []

        # Обработка каждого изображения в батче
        for idx, high_res_target in enumerate(high_res):
            # Получаем метку класса для текущего объекта
            tag = tags[idx].item()  # Извлекаем значение метки как int
            k_list = koefs[tag]  # Получаем соответствующий набор коэффициентов (k1, k2, k3, k4)

            # Разделение изображения на патчи
            patches = split_matrix_into_16(high_res_target[0].cpu().detach().numpy())

            maps = []
            
            # Генерация масок для каждого патча с использованием коэффициентов
            for i in range(len(patches)):
                k = k_list[i // 4]
                map = generate_mask(patches[i], k)
                maps.append(np.array(map))

            # Объединяем патчи обратно в одно изображение
            activation_masks.append(combine_patches_16(maps))

        # Преобразование активационных масок в тензоры
        activation_masks = torch.FloatTensor(activation_masks).to(device).unsqueeze(1)

        optimizer.zero_grad()

        # Прямой проход через UNET
        outputs = model_UNET(high_res)

        # Вычисление потерь
        loss = criterion(outputs, activation_masks)

        # Обратное распространение
        loss.backward()
        optimizer.step()

        # Сохранение значения потерь
        history.append(loss.item())

    print(f'Epoch {epoch+1}, Loss: {history[-1]:.2f}')

In [ ]:
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, targets in train_loader:

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        
        # Прямой проход
        outputs = model(inputs)
        
        # Вычисление потерь
        loss = criterion(outputs, targets)
        
        # Обратное распространение
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        history.append(loss.item())
    
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}')

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(history, label = 'History train')
plt.title('Loss by batch iterations')
plt.ylabel('Error CrossEntropy train dataset')
plt.xlabel('Baches')
#plt.yscale('log')
#plt.xscale('log')
plt.legend()

plt.show()

In [115]:
with torch.no_grad():
    df_test_tensor = df_test_tensor.to(device)
    out = model(df_test_tensor)
    logits = model_UNET(out)


NameError: name 'df_test_tensor' is not defined

In [ ]:
def generate_mask(data, k):

    # Если данные не нормализованы, нормализуйте их
    data = data / np.max(data)

    # Устанавливаем порог на уровне 0.01 от максимального значения
    threshold = k * np.max(data)
    #threshold = 0.00000001 # Старое значение
    #threshold = 0.0001
    # Генерация маски: 1 для сигнала, 0 для фона
    mask = np.where(data > threshold, 1, 0)
    return mask

In [ ]:
cristals = df_test['Crystal']

import DiffQ as DQ

Qplotter = DQ.Qwrapper(960, 500)

for i in range(len(logits)):

    # Создаем фигуру и оси
    fig, (ax1, ax2, ax3) = plt.subplots(1, 2, figsize=(25, 8))
    # Первый график
    Diff = df_test_tensor[i][0].to('cpu').detach().numpy()
    ax1.imshow(Diff, cmap='viridis', aspect='auto', norm='log')
    ax1.set_title(f'Diffraction Low Intensity High Stats {cristals[i]}')
    # Первый график
    Upscaled = out[i][0].to('cpu').detach().numpy()
    Resolution_map = logits[i][0].to('cpu').detach().numpy()
    ax2.imshow(matrix_multiplication_with_threshold(Resolution_map, Upscaled, threshold=0.1), cmap='viridis', aspect='auto', norm='log')
    ax2.set_title(f'UpScaled mask Low Intensity High Stats {cristals[i]}')

    q, I = Qplotter.getQDIFADC(matrix_multiplication_with_threshold(Resolution_map, Upscaled, threshold=0.3), 1024)

    ax3.plot(q, I)

    plt.show()